# Descarga por lote de FITS — Barras

Descarga recortes FITS de tres bandas (`g,r,z`) a partir de `objra` y `objdec`, conservando el identificador MaNGA de cada fila. El límite de descarga es configurable.

In [1]:
from pathlib import Path
import time
import json
import math
import numpy as np
import pandas as pd
import requests

# CONFIGURACIÓN PRINCIPAL
MAX_GALAXIES = 50       # Cambia este número: 50 para prueba, 200, 500, etc.
RANDOM_STATE = 42
CUTOUT_SIZE_PX = 424
PIXEL_SCALE = 0.262
LAYER = "ls-dr10"

# Este descargador usa exclusivamente el CSV de barras ya existente en el proyecto.
# Ubicación esperada: GalMorphAI/BarGalaxiesAnalysis/data_download/dataset.csv
CSV_PATH = Path("../data_download/dataset.csv")
if not CSV_PATH.exists():
    # Permite ejecutarlo si el notebook se abre desde BarGalaxiesAnalysis.
    CSV_PATH = Path("data_download/dataset.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(
        "No se encontró data_download/dataset.csv. Abre este notebook desde "
        "BarGalaxiesAnalysis/feature_engineering o ajusta CSV_PATH."
    )

REQUIRED_COLUMNS = {"name", "objra", "objdec", "Bars", "z"}
headers = set(pd.read_csv(CSV_PATH, nrows=0).columns)
missing_columns = REQUIRED_COLUMNS - headers
if missing_columns:
    raise ValueError(f"El CSV de barras no tiene las columnas requeridas: {sorted(missing_columns)}")

OUTPUT_DIR = CSV_PATH.parent / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = CSV_PATH.parent / "download_report_bars.csv"

print("CSV utilizado:", CSV_PATH.resolve())
print("Carpeta de imágenes:", OUTPUT_DIR.resolve())
print("Máximo de galaxias configurado:", MAX_GALAXIES)

CSV utilizado: C:\Users\VACORTES\Documents\GalMorphAI\BarGalaxiesAnalysis\data_download\dataset.csv
Carpeta de imágenes: C:\Users\VACORTES\Documents\GalMorphAI\BarGalaxiesAnalysis\data_download\data
Máximo de galaxias configurado: 50


## Selección de galaxias

La muestra inicial se balancea por valor de `Bars` para no descargar solamente galaxias sin barra. El número final nunca excede `MAX_GALAXIES`.

In [3]:
df = pd.read_csv(CSV_PATH)
df = df.drop_duplicates(subset="name").dropna(
    subset=["name", "objra", "objdec", "Bars"]
).copy()
df["Bars"] = pd.to_numeric(df["Bars"], errors="raise")

classes = sorted(df["Bars"].unique())
per_class = math.ceil(MAX_GALAXIES / len(classes))

# Se toma una muestra por cada clase sin perder la columna Bars.
selected_parts = [
    group.sample(
        n=min(len(group), per_class),
        random_state=RANDOM_STATE
    )
    for _, group in df.groupby("Bars")
]

selected = (
    pd.concat(selected_parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_STATE)
    .head(MAX_GALAXIES)
    .reset_index(drop=True)
)

print(f"Registros válidos: {len(df)}")
print(f"Galaxias seleccionadas: {len(selected)}")
display(selected["Bars"].value_counts().sort_index().rename("n").to_frame())
display(selected[["name", "objra", "objdec", "Bars"]].head())

Registros válidos: 10126
Galaxias seleccionadas: 50


,n
Bars,
-0.50,9
0.00,8
0.25,9
0.50,8
0.75,8
1.00,8


,name,objra,objdec,Bars
0,manga-11758-6104,202.317541,53.081985,0.25
1,manga-11965-6102,230.749368,8.755389,1.00
2,manga-11958-6104,208.553211,2.880781,1.00
3,manga-8452-3701,156.193567,46.456065,0.00
4,manga-9193-6104,46.749134,-0.811526,0.75


## Descarga de FITS grz

Cada archivo se guarda como `nombre_grz.fits`. La descarga se puede ejecutar más de una vez: los archivos existentes se marcan como `ya_existía` y no se descargan de nuevo.

In [4]:
BASE_URL = "https://www.legacysurvey.org/viewer/fits-cutout"

def build_url(ra, dec):
    return (
        f"{BASE_URL}?ra={float(ra):.8f}&dec={float(dec):.8f}"
        f"&layer={LAYER}&pixscale={PIXEL_SCALE}&bands=grz&size={CUTOUT_SIZE_PX}"
    )

def download_cutout(row):
    name = str(row["name"])
    target = OUTPUT_DIR / f"{name}_grz.fits"
    url = build_url(row["objra"], row["objdec"])
    if target.exists() and target.stat().st_size > 1000:
        return {"name": name, "Bars": row["Bars"], "status": "ya_existía", "file": str(target), "url": url, "detail": ""}
    try:
        response = requests.get(url, timeout=90)
        response.raise_for_status()
        content = response.content
        if not content.startswith(b"SIMPLE"):
            raise ValueError("La respuesta no parece un archivo FITS válido.")
        target.write_bytes(content)
        return {"name": name, "Bars": row["Bars"], "status": "descargado", "file": str(target), "url": url, "detail": f"{len(content)} bytes"}
    except Exception as exc:
        return {"name": name, "Bars": row["Bars"], "status": "error", "file": str(target), "url": url, "detail": str(exc)}

results = []
for position, (_, row) in enumerate(selected.iterrows(), start=1):
    result = download_cutout(row)
    results.append(result)
    print(f"[{position:>3}/{len(selected)}] {result['name']}: {result['status']}")
    time.sleep(0.15)

report = pd.DataFrame(results)
report.to_csv(REPORT_PATH, index=False)
display(report["status"].value_counts().rename("n").to_frame())
display(report.head())

[  1/50] manga-11758-6104: descargado
[  2/50] manga-11965-6102: descargado
[  3/50] manga-11958-6104: descargado
[  4/50] manga-8452-3701: descargado
[  5/50] manga-9193-6104: descargado
[  6/50] manga-8338-1902: descargado
[  7/50] manga-9093-3703: descargado
[  8/50] manga-11868-9102: descargado
[  9/50] manga-8990-3702: descargado
[ 10/50] manga-7495-3704: descargado
[ 11/50] manga-8318-1902: descargado
[ 12/50] manga-8141-3702: descargado
[ 13/50] manga-9891-3702: descargado
[ 14/50] manga-8465-3703: descargado
[ 15/50] manga-9184-12705: descargado
[ 16/50] manga-7957-12703: descargado
[ 17/50] manga-10520-12702: descargado
[ 18/50] manga-8131-6101: descargado
[ 19/50] manga-12087-12703: descargado
[ 20/50] manga-9885-1901: descargado
[ 21/50] manga-8600-12704: descargado
[ 22/50] manga-8615-9101: descargado
[ 23/50] manga-8626-3704: descargado
[ 24/50] manga-11967-6103: descargado
[ 25/50] manga-12683-1902: descargado
[ 26/50] manga-9029-1901: descargado
[ 27/50] manga-11827-3702

,n
status,
descargado,50


,name,Bars,status,file,url,detail
0,manga-11758-6104,0.25,descargado,..\data_download\data\manga-11758-6104_grz.fits,https://www.legacysurvey.org/viewer/fits-cutou...,2162880 bytes
1,manga-11965-6102,1.00,descargado,..\data_download\data\manga-11965-6102_grz.fits,https://www.legacysurvey.org/viewer/fits-cutou...,2162880 bytes
2,manga-11958-6104,1.00,descargado,..\data_download\data\manga-11958-6104_grz.fits,https://www.legacysurvey.org/viewer/fits-cutou...,2162880 bytes
3,manga-8452-3701,0.00,descargado,..\data_download\data\manga-8452-3701_grz.fits,https://www.legacysurvey.org/viewer/fits-cutou...,2162880 bytes
4,manga-9193-6104,0.75,descargado,..\data_download\data\manga-9193-6104_grz.fits,https://www.legacysurvey.org/viewer/fits-cutou...,2162880 bytes


## Verificación y siguientes pasos

El reporte permite conocer qué filas del CSV ya tienen FITS. Solo los registros con estado `descargado` o `ya_existía` podrán alimentar características visuales en la siguiente versión de ingeniería de características.

In [5]:
successful = report[report["status"].isin(["descargado", "ya_existía"])].copy()
coverage = selected[["name", "Bars"]].merge(successful[["name", "status", "file"]], on="name", how="left")
coverage["has_fits"] = coverage["status"].isin(["descargado", "ya_existía"]).astype(int)

coverage_path = CSV_PATH.parent / "bars_fits_coverage.csv"
coverage.to_csv(coverage_path, index=False)

summary = {
    "requested": int(len(selected)),
    "successful": int(coverage["has_fits"].sum()),
    "failed": int((coverage["has_fits"] == 0).sum()),
    "output_dir": str(OUTPUT_DIR),
    "report": str(REPORT_PATH),
    "coverage": str(coverage_path),
    "configuration": {
        "max_galaxies": MAX_GALAXIES,
        "size_px": CUTOUT_SIZE_PX,
        "pixel_scale": PIXEL_SCALE,
        "layer": LAYER
    }
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
display(coverage.head())

{
  "requested": 50,
  "successful": 50,
  "failed": 0,
  "output_dir": "..\\data_download\\data",
  "report": "..\\data_download\\download_report_bars.csv",
  "coverage": "..\\data_download\\bars_fits_coverage.csv",
  "configuration": {
    "max_galaxies": 50,
    "size_px": 424,
    "pixel_scale": 0.262,
    "layer": "ls-dr10"
  }
}


,name,Bars,status,file,has_fits
0,manga-11758-6104,0.25,descargado,..\data_download\data\manga-11758-6104_grz.fits,1
1,manga-11965-6102,1.00,descargado,..\data_download\data\manga-11965-6102_grz.fits,1
2,manga-11958-6104,1.00,descargado,..\data_download\data\manga-11958-6104_grz.fits,1
3,manga-8452-3701,0.00,descargado,..\data_download\data\manga-8452-3701_grz.fits,1
4,manga-9193-6104,0.75,descargado,..\data_download\data\manga-9193-6104_grz.fits,1


**Uso recomendado:** deja `MAX_GALAXIES = 50` para la primera prueba. Revisa el reporte y algunas imágenes. Si la cobertura y calidad son adecuadas, aumenta gradualmente a 200 o más. Las imágenes se descargan desde el servicio público Legacy Surveys usando las coordenadas del CSV.